In [14]:
import pandas as pd
import numpy as np
import pickle
import re

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense

df = pd.read_csv("/content/drive/MyDrive/resume_dataset_1200.csv")

# These lines are removed because the CSV is likely already parsed into 'resume_text' and 'category' columns.
# df[['resume_text','category']] = df['resume_text,category'].str.split(',', n=1, expand=True)
# df = df.drop(columns=['resume_text,category'])

print("Dataset Loaded")
print(df.head())
print(df.columns)

def clean_text(text):
    text = text.lower()
    text = re.sub(r'[^a-zA-Z ]', '', text)
    return text

df["resume_text"] = df["resume_text"].apply(clean_text)

texts = df["resume_text"]
labels = df["category"]

encoder = LabelEncoder()
y = encoder.fit_transform(labels)

tokenizer = Tokenizer(num_words=5000)
tokenizer.fit_on_texts(texts)

sequences = tokenizer.texts_to_sequences(texts)

X = pad_sequences(sequences, maxlen=50)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

model = Sequential()

# Removed input_length as it's deprecated and automatically inferred
model.add(Embedding(5000, 64))
model.add(LSTM(64))
model.add(Dense(32, activation="relu"))
model.add(Dense(len(set(y)), activation="softmax"))

model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

model.fit(X_train, y_train, epochs=20, batch_size=4)

loss, acc = model.evaluate(X_test, y_test)

print("Model Accuracy:", acc)

# Save model in the recommended Keras format
model.save("resume_model.keras")

with open("tokenizer.pkl", "wb") as f:
    pickle.dump(tokenizer, f)

with open("label_encoder.pkl", "wb") as f:
    pickle.dump(encoder, f)

print("Model Saved")

def predict_resume(text):

    text = clean_text(text)

    seq = tokenizer.texts_to_sequences([text])
    padded = pad_sequences(seq, maxlen=50)

    prediction = model.predict(padded)

    label = encoder.inverse_transform([prediction.argmax()])

    return label[0]

resume = input("\nPaste Resume Text:\n")

result = predict_resume(resume)

print("\nPredicted Job Category:", result)

Dataset Loaded
                                         resume_text         category
0  branding content marketing analytics campaign ...        Marketing
1     monitoring automation jenkins ci cd kubernetes  DevOps Engineer
2  user stories product management scrum agile ro...  Product Manager
3  campaign email marketing content marketing ana...        Marketing
4          docker monitoring ci cd kubernetes devops  DevOps Engineer
Index(['resume_text', 'category'], dtype='object')
Epoch 1/20
240/240 ━━━━━━━━━━━━━━━━━━━━ 10s 28ms/step - accuracy: 0.4654 - loss: 1.7340
Epoch 2/20
240/240 ━━━━━━━━━━━━━━━━━━━━ 5s 20ms/step - accuracy: 1.0000 - loss: 0.0140
Epoch 3/20
240/240 ━━━━━━━━━━━━━━━━━━━━ 5s 19ms/step - accuracy: 1.0000 - loss: 0.0037
Epoch 4/20
240/240 ━━━━━━━━━━━━━━━━━━━━ 7s 29ms/step - accuracy: 1.0000 - loss: 0.0020
Epoch 5/20
240/240 ━━━━━━━━━━━━━━━━━━━━ 5s 19ms/step - accuracy: 1.0000 - loss: 0.0012
Epoch 6/20
240/240 ━━━━━━━━━━━━━━━━━━━━ 5s 19ms/step - accuracy: 1.0000 - loss

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 218ms/step

Predicted Job Category: Software Developer
